# 02 · Near-identical images: find them, then rebuild the splits

Why this notebook exists: the first audit (notebook 01, dHash ≤ 20) reported 0% cross-split duplicates,
but the closest pairs it plotted looked like the same leaf 30–34 bits apart (D-004). This notebook
replaces that check with a correlation search over **all** images and rebuilds the splits so that
near-identical images can never be on two sides (D-006).

"Near-identical" means a copy of one photo (re-saved, blurred, flipped, rotated) **or** a re-shot of the
same leaf seconds later. Inspection showed that the second kind is what actually crosses the splits.

Steps: setup → images → stage 1 (32×32, all same-class pairs, 8 rotations/flips) → stage 2 (96×96 check)
→ look at examples → measure the cost of each cut-off → save pairs → rebuild splits → verify → download.

CPU runtime is enough. About 10–15 minutes in total. Result of the 18 Nov 2026 run: cut-off **0.92**.

## 1 · Setup

In [ ]:
import os, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/AnarHummatli/plant-disease-detection.git"
REPO_DIR = "/content/plant-disease-detection"
if os.path.isdir(f"{REPO_DIR}/.git"):
    # the Colab copy is disposable: always match GitHub exactly (a plain pull can be blocked
    # by files that an earlier run of a notebook wrote into the repo folder)
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
%pip install -q -e ".[dev]"
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
os.environ["PLANTDX_ARTIFACTS"] = "/content/drive/MyDrive/plantdx/artifacts"
head = !git -C {REPO_DIR} rev-parse HEAD
remote = !git -C {REPO_DIR} rev-parse origin/main
assert head == remote, "repo is not at the latest commit: run this cell again"
!git -C {REPO_DIR} log -1 --oneline

## 2 · Images and the full image list (all 54,305 images, every split + removed ones)

In [ ]:
import pandas as pd
from plantdx.data.images import download_archive, extract_variant, missing_files

VARIANT, DATA_ROOT = "color", "/content/data"
print("new files extracted:", extract_variant(download_archive(), DATA_ROOT, VARIANT))

parts = [pd.read_csv(f"splits/{s}.csv").assign(split=s) for s in ("train", "val", "test")]
parts.append(pd.read_csv("splits/dropped_train_overlap.csv").assign(split="removed"))
universe = pd.concat(parts, ignore_index=True)
print("images:", len(universe), "| missing on disk:", len(missing_files(DATA_ROOT, VARIANT, universe["path"])))
print(universe["split"].value_counts().to_dict())

## 3 · Stage 1: all same-class pairs at 32×32

`MIN_CORR_STAGE1` only decides which pairs are looked at more closely; it is deliberately low.
If you get *"more than 2,000,000 candidate pairs"*, set it to 0.93 and run the cell again.

In [ ]:
from plantdx.data.duplicates import candidate_pairs, thumbnails
from plantdx.data.images import image_path

MIN_CORR_STAGE1 = 0.90

thumbs = thumbnails([image_path(DATA_ROOT, VARIANT, p) for p in universe["path"]], size=32, workers=os.cpu_count() or 2)
cand = candidate_pairs(universe, thumbs, min_corr=MIN_CORR_STAGE1)
print(f"candidate pairs: {len(cand):,}")
print(cand["transform"].value_counts().to_dict())

## 4 · Stage 2: re-check every candidate at 96×96

In [ ]:
from plantdx.data.duplicates import annotate, load_gray, verify_pairs
from plantdx.paths import artifacts_dir

cand = verify_pairs(cand, lambda p, s: load_gray(image_path(DATA_ROOT, VARIANT, p), s), size=96, workers=os.cpu_count() or 2)
cand = annotate(cand, universe)
raw_dir = artifacts_dir() / "reports_step2"; raw_dir.mkdir(parents=True, exist_ok=True)
cand.sort_values("corr_hi", ascending=False).to_csv(raw_dir / "duplicate_candidates.csv", index=False)
print(cand["corr_hi"].describe(percentiles=[.5, .9, .99]).round(4))

## 5 · Look at examples per correlation band

Each figure shows 6 random pairs **with different leaf ids** from one correlation band (image b is shown
rotated/flipped to match image a).

In [ ]:
from pathlib import Path
from IPython.display import Image as IPImage, display
from plantdx.viz import plot_pairs

FIG = Path("reports/figures"); FIG.mkdir(parents=True, exist_ok=True)
diff_leaf = cand[~cand["same_leaf"]]
BANDS = [(0.90, 0.93), (0.93, 0.95), (0.95, 0.97), (0.97, 0.98), (0.98, 0.99), (0.99, 1.01)]
for lo, hi in BANDS:
    band = diff_leaf[(diff_leaf.corr_hi >= lo) & (diff_leaf.corr_hi < hi)]
    print(f"band {lo:.2f}–{min(hi, 1):.2f}: {len(band):,} pairs with different leaf ids")
    if band.empty:
        continue
    pick = band.sample(min(6, len(band)), random_state=0)
    labels = [f"r={r:.3f} {t}" for r, t in zip(pick["corr_hi"], pick["transform"])]
    out = plot_pairs(pick, DATA_ROOT, VARIANT, f"/content/band_{lo:.2f}.jpg", labels=labels,
                     title=f"correlation {lo:.2f}–{min(hi, 1):.2f}")
    display(IPImage(str(out)))

## 5b · Inspect every pair near the boundary

All pairs ≥ 0.93 and the 12 highest pairs below it. Labels show the correlation and the current splits
(`tr` train, `va` val, `te` test, `rm` removed). In the 18 Nov run about 16 of the 28 pairs ≥ 0.93 and
5–6 of the 12 just below were the same leaf; the rest were different leaves that look alike. So no cut-off
separates them perfectly.

In [ ]:
abbr = {"train": "tr", "val": "va", "test": "te", "removed": "rm"}
above = diff_leaf[diff_leaf.corr_hi >= 0.93].sort_values("corr_hi")
edge = (diff_leaf[(diff_leaf.corr_hi >= 0.915) & (diff_leaf.corr_hi < 0.93)]
        .sort_values("corr_hi", ascending=False).head(12))
for name, df in (("all pairs >= 0.93", above), ("highest 12 below 0.93", edge)):
    for k in range(0, len(df), 7):
        chunk = df.iloc[k:k + 7]
        labels = [f"r={r:.3f} {abbr[a]}/{abbr[b]}" for r, a, b in zip(chunk["corr_hi"], chunk["split_a"], chunk["split_b"])]
        out = plot_pairs(chunk, DATA_ROOT, VARIANT, f"/content/check_{len(name)}_{k}.jpg",
                         labels=labels, title=f"{name}: {k + 1}–{k + len(chunk)} of {len(df)}")
        display(IPImage(str(out)))
print(above.groupby(["split_a", "split_b"]).size())

## 5c · Cost of each cut-off

Merging two different leaves by mistake is harmless (they just end up on the same side); missing a re-shot
of one leaf is leakage. So the cut-off should be **as low as possible** — but below some point look-alike
leaves chain together through union-find into huge groups, and many train images would be removed.
Rule used: the lowest cut-off before chaining starts (largest group stays within about twice the largest
single leaf, 33 images).

In [ ]:
from plantdx.data.split import merge_duplicate_groups

rows = []
for T in (0.90, 0.91, 0.92, 0.93, 0.95):
    p = diff_leaf[diff_leaf.corr_hi >= T]
    u = universe.assign(group=merge_duplicate_groups(universe, p).values)
    test_groups = set(u.loc[u.split == "test", "group"])
    trval = u[u.split.isin(["train", "val"])]
    rows.append({
        "threshold": T,
        "pairs": len(p),
        "cross_split_pairs": int(p.cross_split.sum()),
        "train_val_images_to_remove": int(trval.group.isin(test_groups).sum()),
        "largest_group": int(u.groupby("group").size().max()),
        "largest_leaf": int(u.groupby("leaf_id").size().max()),
        "merged_groups": int((u.groupby("group").leaf_id.nunique() > 1).sum()),
    })
cost = pd.DataFrame(rows)
cost.to_csv("reports/duplicates_threshold_cost.csv", index=False)
display(cost)

## 6 · Save the confirmed pairs

`DUP_THRESHOLD` is the cut-off chosen in 5c (0.92 in the 18 Nov run).

In [ ]:
import json
from datetime import date
from plantdx.data.duplicates import summarize_duplicates
from plantdx.viz import plot_corr_hist

DUP_THRESHOLD = 0.92

confirmed = cand[cand["corr_hi"] >= DUP_THRESHOLD]
keep = confirmed[~confirmed["same_leaf"]]  # same-leaf copies are already in one group
cols = ["path_a", "path_b", "label_index", "corr32", "corr_hi", "transform"]
keep.sort_values(["label_index", "path_a", "path_b"])[cols].to_csv("splits/duplicate_pairs.csv", index=False)

before = summarize_duplicates(confirmed[confirmed["cross_split"]])
print(f"confirmed near-identical pairs: {len(confirmed):,} ({len(keep):,} join different leaf ids)")
print("\nconfirmed pairs across the CURRENT splits (before the fix):")
display(before)

meta = {
    "method": "Pearson correlation of greyscale thumbnails, best of 8 D4 transforms; "
              "stage 1 at 32x32, stage 2 at 96x96 (plantdx.data.duplicates)",
    "stage1_min_corr": MIN_CORR_STAGE1,
    "dup_threshold": DUP_THRESHOLD,
    "candidate_pairs": int(len(cand)),
    "confirmed_pairs": int(len(confirmed)),
    "pairs_joining_different_leaf_ids": int(len(keep)),
    "date": str(date.today()),
}
Path("splits/duplicates_meta.json").write_text(json.dumps(meta, indent=2) + "\n")
before.to_csv("reports/duplicates_before_fix.csv", index=False)
display(IPImage(str(plot_corr_hist(cand, DUP_THRESHOLD, FIG / "dup_corr_hist.png"))))

## 7 · Rebuild the splits with the duplicate groups

In [ ]:
!python -m plantdx.data.prepare

## 8 · Verify

In [ ]:
from plantdx.data.split import assert_no_leakage, cross_split_pairs

new = pd.concat([pd.read_csv(f"splits/{s}.csv").assign(split=s) for s in ("train", "val", "test")], ignore_index=True)
assert_no_leakage(new, pd.read_csv("splits/duplicate_pairs.csv"))
left = cross_split_pairs(new, confirmed)  # ALL confirmed copies, same-leaf ones included
print("confirmed pairs still across splits:", len(left))
assert left.empty

old_split = universe.set_index("path")["split"]
moved = new.assign(old=new["path"].map(old_split)).query("old != split")
removed_now = set(universe["path"]) - set(new["path"])
print("images that changed split:", len(moved), moved.groupby(["old", "split"]).size().to_dict())
print("images removed from train in total (incl. the earlier 290):", len(removed_now))
print(json.dumps(json.load(open("splits/summary.json")), indent=2))

# figures for the report: clearest leaks (highest r) and the cut-off boundary (lowest r)
crossed = confirmed[confirmed["cross_split"] & ~confirmed["same_leaf"]]
for name, pick, title in (
    ("dup_leak_examples.jpg", crossed.nlargest(6, "corr_hi"),
     "Near-identical pairs that crossed the old splits (highest correlations)"),
    ("dup_boundary_pairs.jpg", crossed.nsmallest(6, "corr_hi"),
     f"Boundary of the cut-off (r ≈ {DUP_THRESHOLD}): merged on purpose, may include look-alike leaves"),
):
    if len(pick):
        labels = [f"{a}/{b} r={r:.3f}" for a, b, r in zip(pick["split_a"], pick["split_b"], pick["corr_hi"])]
        display(IPImage(str(plot_pairs(pick, DATA_ROOT, VARIANT, FIG / name, labels=labels, title=title))))

## 9 · Download `splits/` and `reports/`

In [ ]:
import shutil
from google.colab import files

shutil.copytree("reports", artifacts_dir() / "reports_step2", dirs_exist_ok=True)
shutil.copytree("splits", artifacts_dir() / "splits_backup", dirs_exist_ok=True)
!rm -f /content/step2b.zip && zip -qr /content/step2b.zip splits reports
!unzip -l /content/step2b.zip | tail -25
files.download("/content/step2b.zip")